In [3]:
# Check Qiskit, Qiskit Aer versions, Qiskit Machine Learning - Versions
import qiskit
import qiskit_aer
print(qiskit.__version__)
print("Aer:", qiskit_aer.__version__)

2.1.2
Aer: 0.17.2


In [4]:
# --- Library Imports ---
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from scipy.stats import chi2_contingency
from sklearn.model_selection import GridSearchCV, StratifiedKFold
# from imblearn.over_sampling import RandomOverSampler  # Added for optional balancing

In [5]:
# Qiskit Imports
# Definine quantum kernel
# Use the FidelityQuantumKernel class 

from qiskit.circuit.library import ZZFeatureMap
# from qiskit.primitives import StatevectorSampler as Sampler
# from qiskit_machine_learning.state_fidelities import ComputeUncompute
# from qiskit_machine_learning.kernels import FidelityQuantumKernel
# from qiskit_machine_learning.algorithms import QSVC

# from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler, Session
from qiskit import QuantumCircuit, transpile
# from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

In [6]:
# Load data first
lung_cancer_column_names = ['label'] + [f'attr_{i}' for i in range(1, 57)]
file_path_lung = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\lung+cancer\lung-cancer.data'

# reads the data, treating "?" as missing values
df_lung = pd.read_csv(file_path_lung, header=None, names=lung_cancer_column_names, na_values=['?'])
print(f"Original shape of Lung Cancer data: {df_lung.shape}")

Original shape of Lung Cancer data: (32, 57)


In [7]:
# Mode imputation for missing values
modes = df_lung.mode().iloc[0]
df_lung.fillna(modes, inplace=True)

# Then check if all Nan are gone
print(f"Total missing values after imputation: {df_lung.isnull().sum().sum()}\n")

Total missing values after imputation: 0



In [8]:
# Target Binarization
df_lung['label_binary'] = df_lung['label'].apply(lambda x: 0 if x == 1 else 1)

In [9]:
# Check shape again
print("Class distribution (binary):")
print(df_lung['label_binary'].value_counts(normalize=True))

Class distribution (binary):
label_binary
1    0.71875
0    0.28125
Name: proportion, dtype: float64


In [10]:
# Separate Features & Target and Split Data
X_lung = df_lung.drop(['label', 'label_binary'], axis=1)
y_lung_binary = df_lung['label_binary']

In [11]:
X_train_lc, X_test_lc, y_train_lc, y_test_lc = train_test_split(
    X_lung, y_lung_binary, test_size=0.3, random_state=42, stratify=y_lung_binary
)

print(f"Train size: {X_train_lc.shape[0]}, Test size: {X_test_lc.shape[0]}")
print("Train class ratio:", y_train_lc.value_counts(normalize=True).to_dict())

Train size: 22, Test size: 10
Train class ratio: {1: 0.7272727272727273, 0: 0.2727272727272727}


In [12]:
# One-Hot Encoding
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_lc_encoded = pd.DataFrame(encoder.fit_transform(X_train_lc),
columns=encoder.get_feature_names_out())
X_test_lc_encoded = pd.DataFrame(encoder.transform(X_test_lc),
columns=encoder.get_feature_names_out())

In [13]:
# Feature Selection - Cramer's V
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    if min((kcorr-1), (rcorr-1)) == 0: return 0
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

cramers_scores = {col: cramers_v(X_train_lc_encoded[col], y_train_lc) for col in X_train_lc_encoded.columns}
cramers_series = pd.Series(cramers_scores).sort_values(ascending=False)

N_FEATURES_TO_SELECT = 10 
top_features = cramers_series.head(N_FEATURES_TO_SELECT).index.tolist()

# Final Dataframes
X_train_lc_final = X_train_lc_encoded[top_features].to_numpy()
X_test_lc_final = X_test_lc_encoded[top_features].to_numpy()
y_train = y_train_lc.to_numpy()
y_test = y_test_lc.to_numpy()

print("--- Data Preprocessing Complete ---")
print(f"Final training data shape: {X_train_lc_final.shape}")
print(f"Final testing data shape: {X_test_lc_final.shape}\n")


--- Data Preprocessing Complete ---
Final training data shape: (22, 10)
Final testing data shape: (10, 10)



##### Manual Quantum Kernel Implementation

In [14]:
# from qiskit.circuit.library import ZZFeatureMap, unitary_overlap
# from qiskit.primitives import StatevectorSampler # Ideal simulator (Qiskit 1.x compliant)
# If running on hardware, you would import Sampler from qiskit_ibm_runtime or qiskit_quantuminspire

# def compute_kernel_matrix(X1, X2, feature_map, sampler=None):

#    Computes the kernel matrix between X1 and X2 using the overlap circuit.
#    X1: array of shape (n_samples_1, n_features)
#    X2: array of shape (n_samples_2, n_features)

#    num_samples_1 = len(X1)
 #   num_samples_2 = len(X2)
 #   kernel_matrix = np.zeros((num_samples_1, num_samples_2))
    
#    if sampler is None:
 #       sampler = StatevectorSampler()

    # This list will hold all circuits to run in a batch
 #   circuits = []
    
    # 1. Build all overlap circuits
 #   for i in range(num_samples_1):
 #       for j in range(num_samples_2):
            # Bind parameters for data point 1
   #         u1 = feature_map.assign_parameters(X1[i])
            # Bind parameters for data point 2
   #         u2 = feature_map.assign_parameters(X2[j])
            
            # Create overlap: U2_dagger @ U1 |0>
            # Measuring 0 state prob gives |<psi(x2)|psi(x1)>|^2
   #         overlap = unitary_overlap(u1, u2)
   #         overlap.measure_all()
    #        circuits.append(overlap)

    # 2. Run batch execution
    # We run all N*M circuits at once for efficiency
    # For hardware, you might need to chunk this if the job is too large
  #  print(f"Submitting job with {len(circuits)} circuits...")
   # job = sampler.run(circuits, shots=1024) # Adjust shots as needed
    #result = job.result()
    
    # 3. Extract Results and fill Matrix
   # circuit_idx = 0
   # for i in range(num_samples_1):
    #    for j in range(num_samples_2):
            # Get counts for the specific circuit
            # result[idx].data.meas.get_counts() returns a dictionary
     #       data_pub = result[circuit_idx].data.meas.get_counts()
            
            # Calculate probability of measuring '0' (all zeros)
            # Note: Qiskit V2 primitives often return bitstrings. 
            # We check for the all-zero string '0'*num_qubits
     #       all_zeros_key = '0' * feature_map.num_qubits
            
            # If '00..0' wasn't measured, prob is 0.0
    #        zero_count = data_pub.get(all_zeros_key, 0)
      #      total_shots = sum(data_pub.values())
            
      #      kernel_value = zero_count / total_shots
      #      kernel_matrix[i, j] = kernel_value
            
        #    circuit_idx += 1
            
   # return kernel_matrix

# --- Setup Feature Map ---
# feature_dim = X_train_lc_final.shape[1]
# Using ZZFeatureMap as per your original 'ideal' notebook
#fm = ZZFeatureMap(feature_dimension=feature_dim, reps=2, entanglement='linear')

# print("Feature Map created.")

In [15]:
# 1. Setup Backend
from qiskit_quantuminspire.qi_provider import QIProvider
from qiskit.providers import JobStatus

print("\n--- Connecting to Quantum Inspire ---")
try:
    provider = QIProvider()
    available_names = [b.name for b in provider.backends()]
    
    # Prefer QX emulator for testing, Starmon-5 for real hardware
    if 'QX emulator' in available_names:
        backend = provider.get_backend('QX emulator')
    else:
        # Fallback to the first available backend
        backend = provider.get_backend(available_names[0])
        
    print(f"Connected to backend: {backend.name}")

except Exception as e:
    print("Error connecting. Ensure API token is saved.")
    raise e

# 2. Setup Feature Map 
feature_dim = N_FEATURES_TO_SELECT 
fm = ZZFeatureMap(feature_dimension=feature_dim, reps=2, entanglement='linear')
print(f"Feature Map created with {feature_dim} qubits.")


--- Connecting to Quantum Inspire ---
Connected to backend: QX emulator
Feature Map created with 10 qubits.


C:\Users\User\AppData\Local\Temp\ipykernel_5192\2871088292.py:25: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  fm = ZZFeatureMap(feature_dimension=feature_dim, reps=2, entanglement='linear')


In [16]:
def get_job_result_robust(job, check_interval=5):
    """
    Waits for a job to complete by checking status, preventing generic timeouts 
    if the queue is long.
    """
    while True:
        status = job.status()
        if status in [JobStatus.DONE, JobStatus.CANCELLED, JobStatus.ERROR]:
            break
        time.sleep(check_interval)
    
    if status == JobStatus.ERROR:
        raise Exception(f"Job {job.job_id()} failed with error.")
    if status == JobStatus.CANCELLED:
        raise Exception(f"Job {job.job_id()} was cancelled.")
        
    # Once done, retrieve result without a long timeout wait
    return job.result()

def compute_kernel_matrix_optimized(X1, X2, feature_map, backend, batch_size=30):
    """
    Computes Kernel Matrix with:
    1. Symmetry optimization (if X1 is X2).
    2. Batch processing.
    3. Robust job monitoring.
    """
    n1 = len(X1)
    n2 = len(X2)
    
    # Check if we are calculating a symmetric matrix (Train vs Train)
    is_symmetric = np.array_equal(X1, X2)
    
    kernel_matrix = np.zeros((n1, n2))
    
    # 1. Build Circuits
    # We store (row, col) with the circuit to map results back later
    circuits = []
    metadata = [] 
    
    print(f"Building circuits for {n1}x{n2} matrix (Symmetric: {is_symmetric})...")
    
    for i in range(n1):
        # If symmetric, only calculate j >= i (upper triangle)
        start_j = i if is_symmetric else 0
        
        for j in range(start_j, n2):
            qc = QuantumCircuit(feature_map.num_qubits)
            
            # Bind parameters directly
            u1 = feature_map.assign_parameters(X1[i])
            u2 = feature_map.assign_parameters(X2[j])
            
            # Create Compute-Uncompute structure: U(x_i) * U(x_j)^dagger
            qc.compose(u1, inplace=True)
            qc.compose(u2.inverse(), inplace=True)
            qc.measure_all()
            
            circuits.append(qc)
            metadata.append((i, j))

    total_circs = len(circuits)
    print(f"Total circuits to run: {total_circs} (Reduced from {n1*n2})")

    # 2. Transpile & Execute in Batches
    # Transpile all at once locally first (lightweight)
    print("Transpiling circuits locally...")
    t_circuits = transpile(circuits, backend)
    
    zero_state = '0' * feature_map.num_qubits
    
    print(f"Starting execution in batches of {batch_size}...")
    
    current_idx = 0
    while current_idx < total_circs:
        end_idx = min(current_idx + batch_size, total_circs)
        batch = t_circuits[current_idx:end_idx]
        batch_meta = metadata[current_idx:end_idx]
        
        print(f"  > Submitting batch {current_idx} to {end_idx}...")
        
        try:
            job = backend.run(batch, shots=1024)
            # Use custom robust waiter
            result = get_job_result_robust(job)
            counts_list = result.get_counts()
            
            # Qiskit returns a single dict if batch size is 1, list otherwise
            if isinstance(counts_list, dict):
                counts_list = [counts_list]
                
            # Map results to matrix
            for k, counts in enumerate(counts_list):
                row, col = batch_meta[k]
                
                zero_count = counts.get(zero_state, 0)
                total_shots = sum(counts.values())
                prob = zero_count / total_shots
                
                kernel_matrix[row, col] = prob
                
                # If symmetric, mirror the result to the lower triangle
                if is_symmetric and row != col:
                    kernel_matrix[col, row] = prob
                    
        except Exception as e:
            print(f"  ! Error in batch starting at {current_idx}: {e}")
            raise e
            
        current_idx += batch_size
        
    return kernel_matrix

In [17]:
# Compute Kernel Matrices
# sampler = StatevectorSampler() 

# print("Computing Training Kernel Matrix...")
# Symmetrical calculation (Train vs Train)
# matrix_train = compute_kernel_matrix(X_train_lc_final, X_train_lc_final, fm, sampler)

# print("Computing Test Kernel Matrix...")
# Asymmetrical calculation (Test vs Train)
# matrix_test = compute_kernel_matrix(X_test_lc_final, X_train_lc_final, fm, sampler)

# Visualization (Optional)
# plt.figure(figsize=(8, 6))
# plt.imshow(matrix_train, cmap='viridis')
# plt.colorbar()
# plt.title("Manual Quantum Kernel Matrix (Train)")
# plt.show()

In [18]:
# Assuming X_train_lc_final and X_test_lc_final are available from your previous cells

# 1. Compute Train Kernel (Uses Symmetry -> Faster)
print("\n--- Computing Training Kernel ---")
train_kernel = compute_kernel_matrix_optimized(
    X_train_lc_final, 
    X_train_lc_final, 
    fm, 
    backend, 
    batch_size=20 # Smaller batch size to prevent QI limits
)

# 2. Compute Test Kernel (Asymmetric)
print("\n--- Computing Testing Kernel ---")
test_kernel = compute_kernel_matrix_optimized(
    X_test_lc_final, 
    X_train_lc_final, 
    fm, 
    backend, 
    batch_size=20
)

print("\nKernel matrices computation complete.")


--- Computing Training Kernel ---
Building circuits for 22x22 matrix (Symmetric: True)...
Total circuits to run: 253 (Reduced from 484)
Transpiling circuits locally...
Starting execution in batches of 20...
  > Submitting batch 0 to 20...
  > Submitting batch 20 to 40...
  > Submitting batch 40 to 60...
  > Submitting batch 60 to 80...
  > Submitting batch 80 to 100...
  > Submitting batch 100 to 120...
  > Submitting batch 120 to 140...
  > Submitting batch 140 to 160...
  > Submitting batch 160 to 180...
  > Submitting batch 180 to 200...
  > Submitting batch 200 to 220...
  > Submitting batch 220 to 240...
  > Submitting batch 240 to 253...

--- Computing Testing Kernel ---
Building circuits for 10x22 matrix (Symmetric: False)...
Total circuits to run: 220 (Reduced from 220)
Transpiling circuits locally...
Starting execution in batches of 20...
  > Submitting batch 0 to 20...
  > Submitting batch 20 to 40...
  > Submitting batch 40 to 60...
  > Submitting batch 60 to 80...
  > Subm

In [19]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# 1. Initialize SVM with 'precomputed' kernel
qsvc = SVC(kernel='precomputed')

# 2. Fit using the Training Kernel
# Note: X is the (n_samples, n_samples) kernel matrix
qsvc.fit(train_kernel, y_train)

# 3. Predict using the Test Kernel
# Note: X is the (n_test_samples, n_train_samples) kernel matrix
predictions = qsvc.predict(test_kernel)

# 4. Evaluation
print("--- Quantum SVM Results (Quantum Inspire) ---")
print(f"Accuracy: {accuracy_score(y_test, predictions):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, predictions))

--- Quantum SVM Results (Quantum Inspire) ---
Accuracy: 0.70

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.70      1.00      0.82         7

    accuracy                           0.70        10
   macro avg       0.35      0.50      0.41        10
weighted avg       0.49      0.70      0.58        10



c:\Users\User\anaconda3\envs\qi_env\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\User\anaconda3\envs\qi_env\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\User\anaconda3\envs\qi_env\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape